In [5]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [6]:
from pydantic import BaseModel, Field

# === 샘플 문서 준비 ===
document = """Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고,
사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."""

# === 출력 스키마 정의 (선택) ===
# Structured Output 페이지에서 사용한 것과 동일한 구조입니다.
class DocumentAnalysis(BaseModel):
    title: str = Field(description="문서의 제목")
    summary: str = Field(description="문서의 핵심 내용을 2~3문장으로 요약")
    keywords: list[str] = Field(description="문서의 핵심 키워드 3~5개")

In [7]:
from langchain_core.output_parsers import JsonOutputParser

# === 파서 생성 ===
# pydantic_object에 스키마를 지정하면, format_instructions에 필드 설명이 포함됩니다.
parser = JsonOutputParser(pydantic_object=DocumentAnalysis)

# === format_instructions 확인 ===
# LLM에게 전달할 출력 형식 지시를 자동으로 생성합니다.
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY
)

# === 프롬프트 구성 ===
# {format_instructions}에 파서의 형식 지시가 주입됩니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 분석하세요.\n\n{format_instructions}"),
    ("human", "{document}")
])

partial_prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)

# === LCEL 체인 구성 ===
# prompt → model → parser 순서로 연결합니다.
# chain = prompt | model | parser
chain = partial_prompt | model | parser

# === 실행 ===
result = chain.invoke({
    "document": document,
    "format_instructions": parser.get_format_instructions()
})

# === 결과 확인 ===
print(f"타입: {type(result).__name__}")
print(f"제목: {result['title']}")
print(f"요약: {result['summary']}")
print(f"키워드: {result['keywords']}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 49.411344119s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM